[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/38_grpo_loss_solution.ipynb)

# 参考解答：组相对策略优化损失（GRPO）

参考解答。

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✅ 参考解答

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """组相对策略优化（GRPO）损失。

    logps: (B,) 每个采样回复在当前策略下的对数概率
    rewards: (B,) 每个回复的标量奖励
    group_ids: (B,) 整数组编号，相同编号表示同一个提示词／组
    返回：标量损失（Tensor）
    """
    # 计算各组内的归一化优势 A_i
    unique_ids = group_ids.unique()
    advantages = torch.empty_like(rewards)
    for gid in unique_ids:
        mask = group_ids == gid
        r_g = rewards[mask]
        mean_g = r_g.mean()
        std_g = r_g.std(unbiased=False)
        advantages[mask] = (r_g - mean_g) / (std_g + eps)

    # 阻止梯度通过优势传播
    advantages_detached = advantages.detach()

    # GRPO 目标：-E[A_i * logpi_i]
    return -(advantages_detached * logps).mean()


In [ ]:
# 示例
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

In [ ]:
from torch_judge import check
check('grpo_loss')